# DVC Practical Quiz — Data Version Control using DVC

**Name:** Usama Ahsan
**Roll No:** 23F-0029
**University:** FAST, Faisalabad  
**Date:** May 2025

---

**Instructions:**
- No real dataset is required — using dummy data (CSV files).
- All commands are written and explained below.
- Each section corresponds to one quiz question.

---
## Section A — Repository Setup

**Q1.** Create a new project folder called `dvc_lab_quiz` and initialize both Git and DVC inside it.
- Command to create and initialize
- Mention what new folders or files DVC generates

In [ ]:
# Step 1: Create a new project folder (skip if already exists)
import os
os.makedirs('dvc_lab_quiz', exist_ok=True)
os.chdir('dvc_lab_quiz') if os.path.basename(os.getcwd()) != 'dvc_lab_quiz' else None
print(f"Working directory: {os.getcwd()}")

# Step 2: Initialize Git repository
!git init

# Step 3: Initialize DVC inside the Git repo
!dvc init --force

# Step 4: Commit the DVC initialization to Git
!git add .dvc .dvcignore
!git commit -m "Initialize Git and DVC" --allow-empty

### Explanation

When we run `dvc init`, DVC creates the following files and folders:

| File/Folder    | Purpose |
|----------------|----------|
| `.dvc/`        | Internal DVC directory — stores config, cache, and tmp files |
| `.dvc/config`  | DVC configuration file (e.g., remote storage settings) |
| `.dvc/.gitignore` | Tells Git to ignore DVC's internal cache and temp files |
| `.dvcignore`   | Similar to `.gitignore` — tells DVC which files to ignore |

**Project structure after init:**
```
dvc_lab_quiz/
├── .dvc/
│   ├── .gitignore
│   └── config
├── .dvcignore
└── .git/
```

DVC works **on top of Git** — Git handles code and metadata, while DVC handles large data files.

---
## Section B — Adding and Tracking Data

**Q2.** Inside your project, create a folder named `data` and a dummy file `sample.csv` with some random text or numbers. Now, track this file using DVC.
- Write the command to track the data file
- Explain what `.dvc` file is created and what it contains

In [ ]:
import pandas as pd

# Create the data folder
os.makedirs('data', exist_ok=True)

# Create a dummy sample.csv with 10 rows of data
data = {
    'id': range(1, 11),
    'name': [f'Product_{c}' for c in 'ABCDEFGHIJ'],
    'value': [100.5, 205.3, 150.0, 175.8, 220.1, 189.4, 160.0, 195.2, 140.9, 210.5],
    'timestamp': ['2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05',
                  '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10']
}
df = pd.DataFrame(data)
df.to_csv('data/sample.csv', index=False)

print("--- sample.csv contents ---")
print(df.to_string(index=False))

In [ ]:
# Track the data file using DVC
!dvc add data/sample.csv

# Add the generated .dvc file and .gitignore to Git
!git add data/sample.csv.dvc data/.gitignore
!git commit -m "Track sample.csv with DVC" --allow-empty

In [ ]:
# Let's see what the .dvc file looks like
print("--- Contents of data/sample.csv.dvc ---")
with open('data/sample.csv.dvc', 'r') as f:
    print(f.read())

### Explanation

When we run `dvc add data/sample.csv`, DVC does the following:

1. **Computes an MD5 hash** of the file contents
2. **Moves the actual file** into the DVC cache (`.dvc/cache/`)
3. **Creates `data/sample.csv.dvc`** — a small metadata file containing:
   - `md5`: The hash of the file (used for versioning)
   - `size`: File size in bytes
   - `path`: Relative path to the data file
4. **Updates `data/.gitignore`** to tell Git to ignore `sample.csv` (the actual data)

The `.dvc` file is small and gets committed to Git. This way, Git tracks **metadata** (the hash), while DVC tracks the **actual data** — keeping the repo lightweight.

```yaml
# Example contents of sample.csv.dvc:
outs:
- md5: 6998241dec3b40a308276fecb18b25bc
  size: 315
  hash: md5
  path: sample.csv
```

---
## Section C — Remote Storage

**Q3.** Simulate connecting DVC to a remote storage (you can assume a folder `remote_storage` as remote).
- Write the command to add and set the remote
- Mention what the remote does in DVC

In [ ]:
# Create a local folder to act as remote storage
os.makedirs('remote_storage', exist_ok=True)

# Add the folder as a DVC remote and set it as default (-d flag)
# --force in case it already exists
!dvc remote add -d myremote ./remote_storage --force

# Verify the remote was added
!dvc remote list

# Push the tracked data to the remote
!dvc push

### Explanation

A **DVC remote** is like a Git remote, but for data instead of code. It is a storage location where DVC pushes and pulls large data files.

| Command | What it does |
|---------|-------------|
| `dvc remote add -d myremote ./remote_storage` | Adds a local folder as the default remote |
| `dvc remote list` | Lists all configured remotes |
| `dvc push` | Uploads cached data to the remote |
| `dvc pull` | Downloads data from the remote to local cache |

In a real project, the remote would be **cloud storage** like AWS S3, Google Cloud Storage, or Azure Blob. Here we use a local folder for demonstration.

**Why remotes matter:** When a teammate clones the Git repo, they only get the `.dvc` metadata files. They run `dvc pull` to download the actual data from the remote.

---
## Section D — Creating a Simple Pipeline

**Q4.** Suppose you have a script called `process_data.py` that processes your dataset.
- Write a DVC command that defines this script as a pipeline stage that uses `data/sample.csv` and produces an output `processed.csv`.
- Identify which file DVC uses to record pipeline details.

In [ ]:
%%writefile process_data.py
import pandas as pd

# Load data
df = pd.read_csv('data/sample.csv')

# Filter rows where value > 150
df_filtered = df[df['value'] > 150]

# Calculate average value
avg_value = df_filtered['value'].mean()
print(f"Average value of filtered data: {avg_value:.2f}")

# Save processed data
df_filtered.to_csv('processed.csv', index=False)

# Generate metrics
accuracy = round(avg_value / 250, 4)
with open('metrics.txt', 'w') as f:
    f.write(f"accuracy: {accuracy}\n")

print(f"Processing complete. Accuracy: {accuracy}")

In [ ]:
# Define the pipeline stage using dvc stage add
# -n process        : name of the stage
# -d data/sample.csv : dependency (input data)
# -d process_data.py : dependency (script)
# -o processed.csv   : output file
# -m metrics.txt     : metrics file
# --force            : overwrite if stage already exists

!dvc stage add -n process -d data/sample.csv -d process_data.py -o processed.csv -m metrics.txt --force "python process_data.py"

In [ ]:
# Run the pipeline
!dvc repro

# Let's see the generated dvc.yaml
print("\n--- Contents of dvc.yaml ---")
with open('dvc.yaml', 'r') as f:
    print(f.read())

### Explanation

DVC uses **`dvc.yaml`** to record pipeline details. This file defines each stage with:
- **`cmd`** — the command to run (e.g., `python process_data.py`)
- **`deps`** — input dependencies (if these change, the stage reruns)
- **`outs`** — output files produced by the stage
- **`metrics`** — files to track as metrics

```yaml
# dvc.yaml generated by our command:
stages:
  process:
    cmd: python process_data.py
    deps:
    - data/sample.csv
    - process_data.py
    outs:
    - processed.csv
    metrics:
    - metrics.txt
```

DVC also creates a **`dvc.lock`** file which stores the exact hashes of all dependencies and outputs after each run — this ensures reproducibility.

---
## Section E — Tracking and Comparing Metrics

**Q5.** Imagine your project produces a file `metrics.txt` with model accuracy.
- Write commands to show and compare metrics after changes.
- Explain how DVC helps in tracking these metrics across versions.

In [ ]:
# Show current metrics
print("=== Current Metrics ===")
!dvc metrics show

In [ ]:
# Commit the current state so we can compare later
!git add .
!git commit -m "Pipeline v1 with metrics" --allow-empty

In [ ]:
# Now let's modify the data slightly to get different metrics
df_new = pd.read_csv('data/sample.csv')
df_new.loc[0, 'value'] = 230.0   # Change first row value
df_new.loc[1, 'value'] = 250.0   # Change second row value
df_new.to_csv('data/sample.csv', index=False)

# Re-track the modified data
!dvc add data/sample.csv

# Rerun the pipeline (DVC detects the data changed)
!dvc repro

# Show the updated metrics
print("\n=== Updated Metrics ===")
!dvc metrics show

# Compare metrics between current workspace and last commit
print("\n=== Metrics Diff ===")
!dvc metrics diff

### Explanation

DVC tracks metrics through the `metrics` field in `dvc.yaml`. This allows us to:

| Command | Purpose |
|---------|--------|
| `dvc metrics show` | Display the current values of all tracked metrics |
| `dvc metrics diff` | Compare metrics between the current state and the last Git commit |

**How it helps across versions:**
- Every time we `git commit`, the `dvc.lock` file captures the exact metric values.
- We can compare metrics across any two commits using `dvc metrics diff HEAD~1`.
- This lets us track whether model performance is improving or degrading over time, without manually recording numbers.

---
## Section F — Reproduction and Version Control

**Q6.** After changing your dataset slightly, rerun the entire pipeline automatically.
- Write the command to reproduce pipeline stages
- Explain how DVC decides which stages to rerun

In [ ]:
# Reproduce the pipeline
!dvc repro

In [ ]:
# Running it again without changes — DVC will skip all stages
print("=== Running dvc repro again (no changes) ===")
!dvc repro

### Explanation

The command `dvc repro` reproduces the full pipeline by re-executing stages whose dependencies have changed.

**How DVC decides which stages to rerun:**

1. DVC computes the **MD5 hash** of every dependency listed under `deps` in `dvc.yaml`.
2. It compares these hashes with the ones stored in `dvc.lock` (from the last successful run).
3. If **any dependency hash has changed** → that stage and all downstream stages are rerun.
4. If **no hashes have changed** → the stage is skipped (cached result is used).

This is very efficient because:
- Only modified stages are recomputed
- Unchanged stages use cached outputs
- The entire pipeline stays consistent and reproducible

Think of it like **Make** for data science — it only rebuilds what's necessary.

---
## Section G — Final Versioning

**Q7.** Push all updates (data + code) to version control.
- Write the Git and DVC commands for pushing
- Explain how a teammate can exactly reproduce your project results

In [ ]:
# Step 1: Add and commit all changes to Git
!git add .
!git commit -m "Final version with updated data and metrics" --allow-empty

# Step 2: Push data files to DVC remote
!dvc push

# Step 3: Push code + metadata to Git remote
# !git push origin main

### Explanation

The final versioning involves a **two-step push**:

| What | Command | Pushes to |
|------|---------|----------|
| Code + DVC metadata | `git push` | GitHub / GitLab |
| Actual data files | `dvc push` | Remote storage (S3, local folder, etc.) |

### How a teammate reproduces the project:

```bash
# 1. Clone the Git repository
git clone https://github.com/usama-ahsan/dvc_quiz_mlops.git
cd dvc_lab_quiz

# 2. Pull the data from DVC remote
dvc pull

# 3. Reproduce the entire pipeline
dvc repro

# 4. Check the metrics
dvc metrics show
```

This guarantees **exact reproducibility** because:
- Git tracks the code, scripts, `dvc.yaml`, `dvc.lock`, and `.dvc` files
- DVC tracks the exact data versions via hashes
- `dvc repro` re-executes the pipeline using the same data and code
- The teammate gets the exact same results as the original developer

---
## Summary

In this quiz, we covered the full DVC workflow:

| Step | What we did | Key Command |
|------|------------|-------------|
| 1 | Initialized Git + DVC | `git init` + `dvc init` |
| 2 | Tracked dummy data | `dvc add data/sample.csv` |
| 3 | Configured remote storage | `dvc remote add -d myremote ./remote_storage` |
| 4 | Created a pipeline stage | `dvc stage add -n process ...` |
| 5 | Tracked and compared metrics | `dvc metrics show` / `dvc metrics diff` |
| 6 | Reproduced the pipeline | `dvc repro` |
| 7 | Pushed to version control | `git push` + `dvc push` |

**Key takeaway:** DVC extends Git to handle large data and ML pipelines, making data science projects fully reproducible and collaborative.